# ANÁLISIS DE LAS VARIABLES DEL DATASET

In [1]:
import pandas as pd

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test_nolabel.csv")

train.head()

,id,label,statement,subject,speaker,speaker_job,state_info,party_affiliation
0,81f884c64a7,1,China is in the South China Sea and (building)...,"china,foreign-policy,military",donald-trump,President-Elect,New York,republican
1,30c2723a188,0,With the resources it takes to execute just ov...,health-care,chris-dodd,U.S. senator,Connecticut,democrat
2,6936b216e5d,0,The (Wisconsin) governor has proposed tax give...,"corporations,pundits,taxes,abc-news-week",donna-brazile,Political commentator,"Washington, D.C.",democrat
3,b5cd9195738,1,Says her representation of an ex-boyfriend who...,"candidates-biography,children,ethics,families,...",rebecca-bradley,NaN,NaN,none
4,84f8dac7737,0,At protests in Wisconsin against proposed coll...,"health-care,labor,state-budget",republican-party-wisconsin,NaN,Wisconsin,republican


In [2]:
train.info()
train.describe(include='all')

<class 'pandas.DataFrame'>
RangeIndex: 8950 entries, 0 to 8949
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id                 8950 non-null   str  
 1   label              8950 non-null   int64
 2   statement          8950 non-null   str  
 3   subject            8950 non-null   str  
 4   speaker            8950 non-null   str  
 5   speaker_job        6468 non-null   str  
 6   state_info         7020 non-null   str  
 7   party_affiliation  8950 non-null   str  
dtypes: int64(1), str(7)
memory usage: 559.5 KB


,id,label,statement,subject,speaker,speaker_job,state_info,party_affiliation
count,8950,8950.000000,8950,8950,8950,6468,7020,8950
unique,8950,NaN,8939,3409,2634,1090,78,24
top,81f884c64a7,NaN,Martin Luther King Jr. was a Republican.,health-care,barack-obama,President,Texas,republican
freq,1,NaN,2,341,435,438,879,3947
mean,NaN,0.647486,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,0.477780,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
train['label'].value_counts()

label
1    5795
0    3155
Name: count, dtype: int64

El dataset contiene 8950 muestras para una tarea de clasificación binaria, con un ligero desbalanceo hacia la clase positiva. La variable más relevante es statement, que constituye la principal fuente de información. Algunas variables categóricas como party_affiliation y state_info son manejables, mientras que otras como speaker o subject presentan alta cardinalidad o estructura compleja, lo que dificulta su uso directo. Además, existen valores nulos que deberán ser tratados. Como punto de partida, se propone un modelo baseline basado en el procesamiento del texto, al que posteriormente se podrán añadir mejoras.

### Análisis de la variable `id`

La variable `id` actúa como identificador único de cada registro del conjunto de datos.  
No contiene información semántica ni contextual sobre la afirmación, por lo que no se utilizará como característica para entrenar los modelos.

Sin embargo, esta variable se conserva porque será necesaria para construir el archivo final de predicciones que se subirá a Kaggle.

In [5]:
# Comprobación de valores nulos y duplicados en la variable id

print("Nulos en train:", train["id"].isnull().sum())
print("Duplicados en train:", train["id"].duplicated().sum())

print("Nulos en test:", test["id"].isnull().sum())
print("Duplicados en test:", test["id"].duplicated().sum())

# Guardamos los ids del test para la submission final
test_ids = test["id"]

Nulos en train: 0
Duplicados en train: 0
Nulos en test: 0
Duplicados en test: 0


### Análisis de la variable `label`

La variable `label` es la variable objetivo del problema, ya que indica la clase asociada a cada afirmación. Se trata de una tarea de clasificación binaria, por lo que cada muestra pertenece a una de dos posibles clases.

En el conjunto de entrenamiento no existen valores nulos en esta variable. Además, se observa que la distribución de clases no es completamente equilibrada: la clase `1` aparece con mayor frecuencia que la clase `0`.

Por este motivo, en las fases posteriores se tendrá en cuenta este desbalanceo mediante una división estratificada del conjunto de datos y, en algunos modelos, mediante el uso de pesos de clase.

In [6]:
# Análisis de la variable objetivo label

print("Valores únicos:", train["label"].unique())
print("Nulos:", train["label"].isnull().sum())

print("\nDistribución de clases:")
print(train["label"].value_counts())

print("\nDistribución porcentual:")
print(train["label"].value_counts(normalize=True) * 100)

# Guardamos la variable objetivo
y = train["label"]

Valores únicos: [1 0]
Nulos: 0

Distribución de clases:
label
1    5795
0    3155
Name: count, dtype: int64

Distribución porcentual:
label
1    64.748603
0    35.251397
Name: proportion, dtype: float64


## STATEMENT
### Análisis y preparación de la variable `statement`

La variable `statement` contiene el texto principal de cada afirmación y constituye la fuente de información más relevante para el modelo de clasificación. Sobre esta variable se aplican técnicas de procesado de lenguaje natural.

En primer lugar, se comprobó la ausencia de valores nulos y se analizaron posibles frases repetidas. Aunque existen algunas afirmaciones duplicadas, se decidió mantenerlas, ya que pueden aparecer asociadas a diferente contexto o metadatos.

Posteriormente, se aplicó una limpieza básica del texto. Esta limpieza consistió en convertir todas las palabras a minúsculas, eliminar URLs, menciones, hashtags, signos de puntuación, números y espacios duplicados.

Además, se generaron variables adicionales relacionadas con la longitud del texto, como el número de caracteres y el número de palabras de cada afirmación. Estas variables pueden utilizarse posteriormente como características auxiliares para mejorar el rendimiento de los modelos.

In [8]:
import re
import string
import matplotlib.pyplot as plt

# Si no existen todavía, creamos copias
train_prep = train.copy()
test_prep = test.copy()

# Comprobación de nulos
print("Nulos en statement - train:", train_prep["statement"].isnull().sum())
print("Nulos en statement - test:", test_prep["statement"].isnull().sum())

# Comprobación de duplicados
print("Statements duplicados en train:", train_prep["statement"].duplicated().sum())
print("Statements duplicados en test:", test_prep["statement"].duplicated().sum())

# Función de limpieza
def clean_statement(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"@\w+|#\w+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Aplicar limpieza
train_prep["statement_clean"] = train_prep["statement"].apply(clean_statement)
test_prep["statement_clean"] = test_prep["statement"].apply(clean_statement)

# Comparar original y limpio
display(train_prep[["statement", "statement_clean"]].head(10))

# Variables de longitud
train_prep["statement_length"] = train_prep["statement"].apply(len)
test_prep["statement_length"] = test_prep["statement"].apply(len)

train_prep["statement_word_count"] = train_prep["statement"].apply(lambda x: len(str(x).split()))
test_prep["statement_word_count"] = test_prep["statement"].apply(lambda x: len(str(x).split()))

# Estadísticas
display(train_prep[["statement_length", "statement_word_count"]].describe())

# Longitud media por clase
display(train_prep.groupby("label")[["statement_length", "statement_word_count"]].mean())


Nulos en statement - train: 0
Nulos en statement - test: 0
Statements duplicados en train: 11
Statements duplicados en test: 1


,statement,statement_clean
0,China is in the South China Sea and (building)...,china is in the south china sea and buildinga ...
1,With the resources it takes to execute just ov...,with the resources it takes to execute just ov...
2,The (Wisconsin) governor has proposed tax give...,the wisconsin governor has proposed tax giveaw...
3,Says her representation of an ex-boyfriend who...,says her representation of an exboyfriend who ...
4,At protests in Wisconsin against proposed coll...,at protests in wisconsin against proposed coll...
5,Ron Klein sponsored an amendment that specific...,ron klein sponsored an amendment that specific...
6,Rhode Islands legislature is the strongest in ...,rhode islands legislature is the strongest in ...
7,Because of Barack Obama the mission in Iraq en...,because of barack obama the mission in iraq ended
8,"In the last 15 years, weve witnessed a dramati...",in the last years weve witnessed a dramatic ex...
9,The top 1/10th of 1 percent today in America o...,the top th of percent today in america owns al...


,statement_length,statement_word_count
count,8950.000000,8950.000000
mean,107.316313,18.062793
std,59.891256,9.688088
min,11.000000,2.000000
25%,73.000000,12.000000
50%,99.000000,17.000000
75%,133.000000,22.000000
max,3192.000000,467.000000


,statement_length,statement_word_count
label,,
0,108.564184,18.413629
1,106.636928,17.871786


## SUBJECT
### Análisis y preparación de la variable `subject`

La variable `subject` contiene el tema o conjunto de temas asociados a cada afirmación. Esta variable aporta información contextual relevante para el modelo, ya que determinados temas pueden estar más relacionados con una clase que con otra.

En el conjunto de entrenamiento no existen valores nulos en esta columna. Sin embargo, se observa que una misma afirmación puede estar asociada a varios temas, separados por comas. Por este motivo, se creó una lista de temas individuales y una nueva variable con el número de temas asociados a cada afirmación.

Además, se generó una versión limpia de la variable, sustituyendo comas y guiones por espacios, para poder incorporarla posteriormente a la representación textual del modelo.

In [10]:
import re
import matplotlib.pyplot as plt

# Comprobación de nulos
print("Nulos en subject - train:", train_prep["subject"].isnull().sum())
print("Nulos en subject - test:", test_prep["subject"].isnull().sum())

# Valores más frecuentes
print("Combinaciones de subject más frecuentes:")
display(train_prep["subject"].value_counts().head(20))

# Crear lista de subjects
train_prep["subject_list"] = train_prep["subject"].apply(lambda x: str(x).split(","))
test_prep["subject_list"] = test_prep["subject"].apply(lambda x: str(x).split(","))

# Número de subjects por afirmación
train_prep["num_subjects"] = train_prep["subject_list"].apply(len)
test_prep["num_subjects"] = test_prep["subject_list"].apply(len)

# Función de limpieza
def clean_subject(text):
    text = str(text).lower()
    text = text.replace(",", " ")
    text = text.replace("-", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Aplicar limpieza
train_prep["subject_clean"] = train_prep["subject"].apply(clean_subject)
test_prep["subject_clean"] = test_prep["subject"].apply(clean_subject)

# Comprobar resultado
display(train_prep[["subject", "subject_list", "num_subjects", "subject_clean"]].head(10))

# Temas individuales más frecuentes
all_subjects = train_prep["subject_list"].explode()

print("Temas individuales más frecuentes:")
display(all_subjects.value_counts().head(20))

# Media de número de temas por clase
print("Número medio de temas por clase:")
display(train_prep.groupby("label")["num_subjects"].mean())


Nulos en subject - train: 0
Nulos en subject - test: 0
Combinaciones de subject más frecuentes:


subject
health-care               341
taxes                     244
elections                 225
immigration               224
education                 201
candidates-biography      166
economy                   121
guns                      104
economy,jobs              104
federal-budget             99
jobs                       93
abortion                   86
energy                     81
foreign-policy             77
state-budget               73
transportation             67
education,state-budget     55
ethics                     54
environment                53
iraq                       51
Name: count, dtype: int64

,subject,subject_list,num_subjects,subject_clean
0,"china,foreign-policy,military","[china, foreign-policy, military]",3,china foreign policy military
1,health-care,[health-care],1,health care
2,"corporations,pundits,taxes,abc-news-week","[corporations, pundits, taxes, abc-news-week]",4,corporations pundits taxes abc news week
3,"candidates-biography,children,ethics,families,...","[candidates-biography, children, ethics, famil...",5,candidates biography children ethics families ...
4,"health-care,labor,state-budget","[health-care, labor, state-budget]",3,health care labor state budget
5,candidates-biography,[candidates-biography],1,candidates biography
6,"legal-issues,state-budget,states","[legal-issues, state-budget, states]",3,legal issues state budget states
7,iraq,[iraq],1,iraq
8,immigration,[immigration],1,immigration
9,"economy,income,retirement,social-security,wealth","[economy, income, retirement, social-security,...",5,economy income retirement social security wealth


Temas individuales más frecuentes:


subject_list
economy                 997
health-care             991
taxes                   857
federal-budget          646
education               638
jobs                    632
state-budget            612
candidates-biography    572
elections               536
immigration             456
foreign-policy          410
crime                   384
history                 352
energy                  328
legal-issues            304
environment             299
guns                    283
military                266
job-accomplishments     261
workers                 247
Name: count, dtype: int64

Número medio de temas por clase:


label
0    2.103645
1    2.180846
Name: num_subjects, dtype: float64

## SPEAKER
### Análisis y preparación de la variable `speaker`

La variable `speaker` identifica a la persona, partido u organización que realiza cada afirmación. Esta variable puede aportar información contextual relevante, ya que algunos hablantes aparecen de forma recurrente en el conjunto de datos.

En el conjunto de entrenamiento no se observan valores nulos en esta variable. Sin embargo, existe una gran variedad de hablantes distintos, por lo que se generaron varias transformaciones. En primer lugar, se creó una versión limpia de la variable, sustituyendo guiones por espacios y convirtiendo el texto a minúsculas. Además, se calculó la frecuencia de aparición de cada hablante y se creó una versión agrupada, donde los hablantes con pocas apariciones se sustituyen por la categoría `other`.

Estas transformaciones permiten conservar información relevante sobre el hablante, reduciendo al mismo tiempo el ruido producido por categorías muy poco frecuentes.

In [12]:
import re
import matplotlib.pyplot as plt

# Comprobación de nulos
print("Nulos en speaker - train:", train_prep["speaker"].isnull().sum())
print("Nulos en speaker - test:", test_prep["speaker"].isnull().sum())

# Número de speakers únicos
print("Speakers únicos en train:", train_prep["speaker"].nunique())
print("Speakers únicos en test:", test_prep["speaker"].nunique())

# Speakers más frecuentes
print("Speakers más frecuentes:")
display(train_prep["speaker"].value_counts().head(20))

# Limpieza de speaker
def clean_speaker(text):
    text = str(text).lower()
    text = text.replace("-", " ")
    text = text.replace("_", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_prep["speaker_clean"] = train_prep["speaker"].apply(clean_speaker)
test_prep["speaker_clean"] = test_prep["speaker"].apply(clean_speaker)

# Frecuencia de speaker calculada solo con train
speaker_freq = train_prep["speaker"].value_counts()

train_prep["speaker_freq"] = train_prep["speaker"].map(speaker_freq)
test_prep["speaker_freq"] = test_prep["speaker"].map(speaker_freq)

# Speakers del test que no aparecen en train tendrán frecuencia 0
test_prep["speaker_freq"] = test_prep["speaker_freq"].fillna(0)

# Agrupar speakers poco frecuentes
frequent_speakers = speaker_freq[speaker_freq >= 3].index

train_prep["speaker_grouped"] = train_prep["speaker"].apply(
    lambda x: x if x in frequent_speakers else "other"
)

test_prep["speaker_grouped"] = test_prep["speaker"].apply(
    lambda x: x if x in frequent_speakers else "other"
)

train_prep["speaker_grouped_clean"] = train_prep["speaker_grouped"].apply(clean_speaker)
test_prep["speaker_grouped_clean"] = test_prep["speaker_grouped"].apply(clean_speaker)

# Comprobar resultado
display(train_prep[[
    "speaker",
    "speaker_clean",
    "speaker_freq",
    "speaker_grouped",
    "speaker_grouped_clean"
]].head(10))

# Estadísticas por speaker
speaker_stats = train_prep.groupby("speaker").agg(
    total=("label", "count"),
    mean_label=("label", "mean")
).sort_values("total", ascending=False)

print("Estadísticas de los speakers más frecuentes:")
display(speaker_stats.head(20))

# Speakers con al menos 10 apariciones
speaker_stats_10 = speaker_stats[speaker_stats["total"] >= 10]

print("Speakers con mayor proporción de label=1:")
display(speaker_stats_10.sort_values("mean_label", ascending=False).head(10))

print("Speakers con menor proporción de label=1:")
display(speaker_stats_10.sort_values("mean_label", ascending=True).head(10))


Nulos en speaker - train: 0
Nulos en speaker - test: 0
Speakers únicos en train: 2634
Speakers únicos en test: 1511
Speakers más frecuentes:


speaker
barack-obama       435
donald-trump       247
hillary-clinton    204
mitt-romney        142
john-mccain        139
chain-email        128
scott-walker       123
rick-perry         117
rick-scott         105
marco-rubio         99
ted-cruz            82
bernie-s            75
facebook-posts      75
chris-christie      74
charlie-crist       62
newt-gingrich       62
jeb-bush            60
blog-posting        57
joe-biden           54
paul-ryan           50
Name: count, dtype: int64

,speaker,speaker_clean,speaker_freq,speaker_grouped,speaker_grouped_clean
0,donald-trump,donald trump,247,donald-trump,donald trump
1,chris-dodd,chris dodd,7,chris-dodd,chris dodd
2,donna-brazile,donna brazile,9,donna-brazile,donna brazile
3,rebecca-bradley,rebecca bradley,2,other,other
4,republican-party-wisconsin,republican party wisconsin,10,republican-party-wisconsin,republican party wisconsin
5,we-love-usa-pac,we love usa pac,1,other,other
6,lincoln-chafee,lincoln chafee,11,lincoln-chafee,lincoln chafee
7,barack-obama,barack obama,435,barack-obama,barack obama
8,aclu-georgia-foundation,aclu georgia foundation,1,other,other
9,bernie-s,bernie s,75,bernie-s,bernie s


Estadísticas de los speakers más frecuentes:


,total,mean_label
speaker,,
barack-obama,435,0.535632
donald-trump,247,0.834008
hillary-clinton,204,0.480392
mitt-romney,142,0.718310
john-mccain,139,0.589928
chain-email,128,0.968750
scott-walker,123,0.650407
rick-perry,117,0.717949
rick-scott,105,0.647619


Speakers con mayor proporción de label=1:


,total,mean_label
speaker,,
american-crossroads,10,1.000000
americans-prosperity,15,1.000000
viral-image,18,1.000000
chain-email,128,0.968750
rush-limbaugh,25,0.960000
national-republican-senatorial-committee,24,0.958333
democratic-congressional-campaign-committee,21,0.952381
ben-carson,19,0.947368
martin-omalley,11,0.909091


Speakers con menor proporción de label=1:


,total,mean_label
speaker,,
cory-booker,12,0.250000
alex-sink,15,0.266667
julian-castro,11,0.272727
jim-moran,10,0.300000
jeff-merkley,10,0.300000
bill-richardson,10,0.300000
bobby-scott,10,0.300000
sheldon-whitehouse,18,0.333333
dennis-kucinich,17,0.352941


## SPEAKER_JOB
### Análisis y preparación de la variable `speaker_job`

La variable `speaker_job` contiene la profesión o cargo asociado al hablante que realiza la afirmación. Esta información puede resultar útil para el modelo, ya que aporta contexto adicional sobre el origen de la declaración.

En el análisis inicial se observó que esta variable contenía un número elevado de valores nulos. En lugar de eliminar estas muestras, se decidió sustituir los valores ausentes por la categoría `unknown`, conservando así todos los registros del conjunto de datos.

Posteriormente, se generó una versión limpia de la variable, convirtiendo el texto a minúsculas y sustituyendo símbolos como guiones, puntos y comas por espacios. Además, debido al alto número de cargos distintos, se calculó la frecuencia de aparición de cada cargo y se creó una versión agrupada, donde los cargos poco frecuentes fueron sustituidos por la categoría `other`.

In [14]:
import re
import matplotlib.pyplot as plt

# Comprobación de nulos
print("Nulos en speaker_job - train:", train_prep["speaker_job"].isnull().sum())
print("Nulos en speaker_job - test:", test_prep["speaker_job"].isnull().sum())

# Rellenar nulos
train_prep["speaker_job"] = train_prep["speaker_job"].fillna("unknown")
test_prep["speaker_job"] = test_prep["speaker_job"].fillna("unknown")

# Comprobar después de rellenar
print("Nulos en speaker_job - train tras rellenar:", train_prep["speaker_job"].isnull().sum())
print("Nulos en speaker_job - test tras rellenar:", test_prep["speaker_job"].isnull().sum())

# Valores más frecuentes
print("Speaker jobs más frecuentes:")
display(train_prep["speaker_job"].value_counts().head(20))

# Número de cargos únicos
print("Speaker jobs únicos en train:", train_prep["speaker_job"].nunique())
print("Speaker jobs únicos en test:", test_prep["speaker_job"].nunique())

# Función de limpieza
def clean_speaker_job(text):
    text = str(text).lower()
    text = text.replace("-", " ")
    text = text.replace("_", " ")
    text = text.replace(".", " ")
    text = text.replace(",", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Aplicar limpieza
train_prep["speaker_job_clean"] = train_prep["speaker_job"].apply(clean_speaker_job)
test_prep["speaker_job_clean"] = test_prep["speaker_job"].apply(clean_speaker_job)

# Frecuencia del cargo calculada solo con train
speaker_job_freq = train_prep["speaker_job"].value_counts()

train_prep["speaker_job_freq"] = train_prep["speaker_job"].map(speaker_job_freq)
test_prep["speaker_job_freq"] = test_prep["speaker_job"].map(speaker_job_freq)

# En test puede haber cargos no vistos en train
test_prep["speaker_job_freq"] = test_prep["speaker_job_freq"].fillna(0)

# Agrupar cargos poco frecuentes
frequent_jobs = speaker_job_freq[speaker_job_freq >= 3].index

train_prep["speaker_job_grouped"] = train_prep["speaker_job"].apply(
    lambda x: x if x in frequent_jobs else "other"
)

test_prep["speaker_job_grouped"] = test_prep["speaker_job"].apply(
    lambda x: x if x in frequent_jobs else "other"
)

train_prep["speaker_job_grouped_clean"] = train_prep["speaker_job_grouped"].apply(clean_speaker_job)
test_prep["speaker_job_grouped_clean"] = test_prep["speaker_job_grouped"].apply(clean_speaker_job)

# Comprobar resultado
display(train_prep[[
    "speaker_job",
    "speaker_job_clean",
    "speaker_job_freq",
    "speaker_job_grouped",
    "speaker_job_grouped_clean"
]].head(10))

# Estadísticas por cargo
speaker_job_stats = train_prep.groupby("speaker_job").agg(
    total=("label", "count"),
    mean_label=("label", "mean")
).sort_values("total", ascending=False)

print("Estadísticas de los cargos más frecuentes:")
display(speaker_job_stats.head(20))

# Cargos con al menos 10 apariciones
speaker_job_stats_10 = speaker_job_stats[speaker_job_stats["total"] >= 10]

print("Cargos con mayor proporción de label=1:")
display(speaker_job_stats_10.sort_values("mean_label", ascending=False).head(10))

print("Cargos con menor proporción de label=1:")
display(speaker_job_stats_10.sort_values("mean_label", ascending=True).head(10))


Nulos en speaker_job - train: 0
Nulos en speaker_job - test: 0
Nulos en speaker_job - train tras rellenar: 0
Nulos en speaker_job - test tras rellenar: 0
Speaker jobs más frecuentes:


speaker_job
unknown                          2482
President                         438
U.S. Senator                      391
Governor                          335
President-Elect                   247
U.S. senator                      236
Presidential candidate            215
U.S. Representative               149
Former governor                   142
Senator                           129
Milwaukee County Executive        123
State Senator                     104
U.S. representative                81
U.S. House of Representatives      81
Attorney                           78
Social media posting               75
Governor of New Jersey             74
Co-host on CNN's "Crossfire"       66
State Representative               63
Congressman                        61
Name: count, dtype: int64

Speaker jobs únicos en train: 1091
Speaker jobs únicos en test: 635


,speaker_job,speaker_job_clean,speaker_job_freq,speaker_job_grouped,speaker_job_grouped_clean
0,President-Elect,president elect,247,President-Elect,president elect
1,U.S. senator,u s senator,236,U.S. senator,u s senator
2,Political commentator,political commentator,15,Political commentator,political commentator
3,unknown,unknown,2482,unknown,unknown
4,unknown,unknown,2482,unknown,unknown
5,unknown,unknown,2482,unknown,unknown
6,unknown,unknown,2482,unknown,unknown
7,President,president,438,President,president
8,unknown,unknown,2482,unknown,unknown
9,U.S. Senator,u s senator,391,U.S. Senator,u s senator


Estadísticas de los cargos más frecuentes:


,total,mean_label
speaker_job,,
unknown,2482,0.697421
President,438,0.536530
U.S. Senator,391,0.537084
Governor,335,0.665672
President-Elect,247,0.834008
U.S. senator,236,0.567797
Presidential candidate,215,0.493023
U.S. Representative,149,0.664430
Former governor,142,0.718310


Cargos con mayor proporción de label=1:


,total,mean_label
speaker_job,,
Radio host,26,0.961538
political action committee,13,0.923077
Milwaukee County Sheriff,11,0.909091
Maryland governor,11,0.909091
Advocacy group,10,0.900000
Member of the U.S. House,15,0.866667
"Chairman, Republican National Committee",19,0.842105
President-Elect,247,0.834008
Philanthropist,18,0.833333


Cargos con menor proporción de label=1:


,total,mean_label
speaker_job,,
"Mayor, San Antonio",11,0.272727
U. S. Congressman,10,0.300000
U.S. senator from Ohio,39,0.384615
State senator,41,0.390244
Governor of Oregon,10,0.400000
venture capital company founder,10,0.400000
Texas House Representative,15,0.400000
Actor,15,0.466667
"Governor of Ohio as of Jan. 10, 2011",45,0.488889


## STATE_INFO
### Análisis y preparación de la variable `state_info`

La variable `state_info` contiene información geográfica relacionada con la afirmación o con el hablante. Esta variable puede aportar contexto adicional al modelo, especialmente en afirmaciones vinculadas a estados o regiones concretas.

En el análisis inicial se observó que esta columna contenía valores nulos. Para evitar la eliminación de registros, dichos valores se sustituyeron por la categoría `unknown`.

Posteriormente, se generó una versión limpia de la variable, convirtiendo el texto a minúsculas y sustituyendo símbolos como comas, puntos y guiones por espacios. Además, se calculó la frecuencia de aparición de cada estado dentro del conjunto de entrenamiento, creando una característica adicional que representa la presencia de cada contexto geográfico en los datos.

In [15]:
import re
import matplotlib.pyplot as plt

# Comprobación de nulos
print("Nulos en state_info - train:", train_prep["state_info"].isnull().sum())
print("Nulos en state_info - test:", test_prep["state_info"].isnull().sum())

# Rellenar nulos
train_prep["state_info"] = train_prep["state_info"].fillna("unknown")
test_prep["state_info"] = test_prep["state_info"].fillna("unknown")

# Comprobar después de rellenar
print("Nulos en state_info - train tras rellenar:", train_prep["state_info"].isnull().sum())
print("Nulos en state_info - test tras rellenar:", test_prep["state_info"].isnull().sum())

# Valores más frecuentes
print("Estados más frecuentes:")
display(train_prep["state_info"].value_counts().head(20))

# Número de estados únicos
print("Estados únicos en train:", train_prep["state_info"].nunique())
print("Estados únicos en test:", test_prep["state_info"].nunique())

# Función de limpieza
def clean_state(text):
    text = str(text).lower()
    text = text.replace("-", " ")
    text = text.replace("_", " ")
    text = text.replace(".", " ")
    text = text.replace(",", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Aplicar limpieza
train_prep["state_info_clean"] = train_prep["state_info"].apply(clean_state)
test_prep["state_info_clean"] = test_prep["state_info"].apply(clean_state)

# Frecuencia del estado calculada solo con train
state_freq = train_prep["state_info"].value_counts()

train_prep["state_info_freq"] = train_prep["state_info"].map(state_freq)
test_prep["state_info_freq"] = test_prep["state_info"].map(state_freq)

# En test puede haber estados no vistos en train
test_prep["state_info_freq"] = test_prep["state_info_freq"].fillna(0)

# Comprobar resultado
display(train_prep[[
    "state_info",
    "state_info_clean",
    "state_info_freq"
]].head(10))

# Estadísticas por estado
state_stats = train_prep.groupby("state_info").agg(
    total=("label", "count"),
    mean_label=("label", "mean")
).sort_values("total", ascending=False)

print("Estadísticas de los estados más frecuentes:")
display(state_stats.head(20))

# Estados con al menos 10 apariciones
state_stats_10 = state_stats[state_stats["total"] >= 10]

print("Estados con mayor proporción de label=1:")
display(state_stats_10.sort_values("mean_label", ascending=False).head(10))

print("Estados con menor proporción de label=1:")
display(state_stats_10.sort_values("mean_label", ascending=True).head(10))


Nulos en state_info - train: 1930
Nulos en state_info - test: 818
Nulos en state_info - train tras rellenar: 0
Nulos en state_info - test tras rellenar: 0
Estados más frecuentes:


state_info
unknown             1930
Texas                879
Florida              853
Wisconsin            648
New York             579
Illinois             487
Ohio                 408
Georgia              381
Virginia             368
Rhode Island         317
Oregon               220
New Jersey           209
Massachusetts        167
Arizona              160
California           121
Washington, D.C.      89
Vermont               80
New Hampshire         79
Pennsylvania          79
Arkansas              77
Name: count, dtype: int64

Estados únicos en train: 79
Estados únicos en test: 67


,state_info,state_info_clean,state_info_freq
0,New York,new york,579
1,Connecticut,connecticut,21
2,"Washington, D.C.",washington d c,89
3,unknown,unknown,1930
4,Wisconsin,wisconsin,648
5,unknown,unknown,1930
6,Rhode Island,rhode island,317
7,Illinois,illinois,487
8,unknown,unknown,1930
9,Vermont,vermont,80


Estadísticas de los estados más frecuentes:


,total,mean_label
state_info,,
unknown,1930,0.717617
Texas,879,0.654152
Florida,853,0.617819
Wisconsin,648,0.703704
New York,579,0.673575
Illinois,487,0.554415
Ohio,408,0.517157
Georgia,381,0.608924
Virginia,368,0.619565


Estados con mayor proporción de label=1:


,total,mean_label
state_info,,
Iowa,13,0.846154
Indiana,38,0.815789
Alabama,10,0.800000
"Washington, D.C.",10,0.800000
Pennsylvania,79,0.784810
Minnesota,54,0.777778
Alaska,50,0.760000
Utah,19,0.736842
Colorado,22,0.727273


Estados con menor proporción de label=1:


,total,mean_label
state_info,,
Connecticut,21,0.238095
Vermont,80,0.437500
Ohio,408,0.517157
Illinois,487,0.554415
North Carolina,51,0.568627
Michigan,21,0.571429
New Jersey,209,0.578947
New Mexico,24,0.583333
California,121,0.586777


## PARTY_AFFILIATION
### Análisis y preparación de la variable `party_affiliation`

La variable `party_affiliation` indica la afiliación política asociada al hablante que realiza la afirmación. Esta variable aporta información contextual relevante para el problema, ya que el dataset está relacionado con afirmaciones políticas.

En el conjunto de entrenamiento no se observan valores nulos en esta columna. Además, se identifican 24 categorías distintas, siendo `republican` y `democrat` algunas de las más frecuentes.

Para preparar esta variable, se creó una versión limpia convirtiendo el texto a minúsculas y sustituyendo guiones o barras bajas por espacios. También se calculó la frecuencia de aparición de cada afiliación política en el conjunto de entrenamiento, generando así una característica adicional que puede ser utilizada por los modelos.

In [16]:
import re
import matplotlib.pyplot as plt

# Comprobación de nulos
print("Nulos en party_affiliation - train:", train_prep["party_affiliation"].isnull().sum())
print("Nulos en party_affiliation - test:", test_prep["party_affiliation"].isnull().sum())

# Rellenar posibles nulos
train_prep["party_affiliation"] = train_prep["party_affiliation"].fillna("unknown")
test_prep["party_affiliation"] = test_prep["party_affiliation"].fillna("unknown")

# Valores más frecuentes
print("Afiliaciones políticas más frecuentes:")
display(train_prep["party_affiliation"].value_counts())

print("Afiliaciones políticas en porcentaje:")
display(train_prep["party_affiliation"].value_counts(normalize=True) * 100)

# Número de categorías
print("Partidos únicos en train:", train_prep["party_affiliation"].nunique())
print("Partidos únicos en test:", test_prep["party_affiliation"].nunique())

# Función de limpieza
def clean_party(text):
    text = str(text).lower()
    text = text.replace("-", " ")
    text = text.replace("_", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Aplicar limpieza
train_prep["party_affiliation_clean"] = train_prep["party_affiliation"].apply(clean_party)
test_prep["party_affiliation_clean"] = test_prep["party_affiliation"].apply(clean_party)

# Frecuencia calculada solo con train
party_freq = train_prep["party_affiliation"].value_counts()

train_prep["party_affiliation_freq"] = train_prep["party_affiliation"].map(party_freq)
test_prep["party_affiliation_freq"] = test_prep["party_affiliation"].map(party_freq)

# En test puede haber categorías no vistas en train
test_prep["party_affiliation_freq"] = test_prep["party_affiliation_freq"].fillna(0)

# Comprobar resultado
display(train_prep[[
    "party_affiliation",
    "party_affiliation_clean",
    "party_affiliation_freq"
]].head(10))

# Estadísticas por afiliación política
party_stats = train_prep.groupby("party_affiliation").agg(
    total=("label", "count"),
    mean_label=("label", "mean")
).sort_values("total", ascending=False)

print("Relación entre party_affiliation y label:")
display(party_stats)


Nulos en party_affiliation - train: 0
Nulos en party_affiliation - test: 0
Afiliaciones políticas más frecuentes:


party_affiliation
republican                      3947
democrat                        2898
none                            1531
organization                     197
independent                      130
newsmaker                         41
journalist                        37
libertarian                       34
columnist                         32
activist                          30
talk-show-host                    23
state-official                    14
tea-party-member                   8
business-leader                    8
labor-leader                       7
green                              3
constitution-party                 2
education-official                 2
Moderate                           1
county-commissioner                1
liberal-party-canada               1
ocean-state-tea-party-action       1
government-body                    1
democratic-farmer-labor            1
Name: count, dtype: int64

Afiliaciones políticas en porcentaje:


party_affiliation
republican                      44.100559
democrat                        32.379888
none                            17.106145
organization                     2.201117
independent                      1.452514
newsmaker                        0.458101
journalist                       0.413408
libertarian                      0.379888
columnist                        0.357542
activist                         0.335196
talk-show-host                   0.256983
state-official                   0.156425
tea-party-member                 0.089385
business-leader                  0.089385
labor-leader                     0.078212
green                            0.033520
constitution-party               0.022346
education-official               0.022346
Moderate                         0.011173
county-commissioner              0.011173
liberal-party-canada             0.011173
ocean-state-tea-party-action     0.011173
government-body                  0.011173
democratic-farme

Partidos únicos en train: 24
Partidos únicos en test: 19


,party_affiliation,party_affiliation_clean,party_affiliation_freq
0,republican,republican,3947
1,democrat,democrat,2898
2,democrat,democrat,2898
3,none,none,1531
4,republican,republican,3947
5,none,none,1531
6,democrat,democrat,2898
7,democrat,democrat,2898
8,none,none,1531
9,independent,independent,130


Relación entre party_affiliation y label:


,total,mean_label
party_affiliation,,
republican,3947,0.695212
democrat,2898,0.562112
none,1531,0.693664
organization,197,0.812183
independent,130,0.446154
newsmaker,41,0.536585
journalist,37,0.513514
libertarian,34,0.617647
columnist,32,0.593750


In [17]:
train_prep.to_csv("../data/train_preprocessed.csv", index=False)
test_prep.to_csv("../data/test_preprocessed.csv", index=False)

In [18]:
print("Train:", train_prep.shape)
print("Test:", test_prep.shape)

display(train_prep.head())
display(test_prep.head())

Train: (8950, 26)
Test: (3836, 25)


,id,label,statement,subject,speaker,speaker_job,state_info,party_affiliation,statement_clean,statement_length,...,speaker_grouped,speaker_grouped_clean,speaker_job_clean,speaker_job_freq,speaker_job_grouped,speaker_job_grouped_clean,state_info_clean,state_info_freq,party_affiliation_clean,party_affiliation_freq
0,81f884c64a7,1,China is in the South China Sea and (building)...,"china,foreign-policy,military",donald-trump,President-Elect,New York,republican,china is in the south china sea and buildinga ...,116,...,donald-trump,donald trump,president elect,247,President-Elect,president elect,new york,579,republican,3947
1,30c2723a188,0,With the resources it takes to execute just ov...,health-care,chris-dodd,U.S. senator,Connecticut,democrat,with the resources it takes to execute just ov...,164,...,chris-dodd,chris dodd,u s senator,236,U.S. senator,u s senator,connecticut,21,democrat,2898
2,6936b216e5d,0,The (Wisconsin) governor has proposed tax give...,"corporations,pundits,taxes,abc-news-week",donna-brazile,Political commentator,"Washington, D.C.",democrat,the wisconsin governor has proposed tax giveaw...,68,...,donna-brazile,donna brazile,political commentator,15,Political commentator,political commentator,washington d c,89,democrat,2898
3,b5cd9195738,1,Says her representation of an ex-boyfriend who...,"candidates-biography,children,ethics,families,...",rebecca-bradley,unknown,unknown,none,says her representation of an exboyfriend who ...,135,...,other,other,unknown,2482,unknown,unknown,unknown,1930,none,1531
4,84f8dac7737,0,At protests in Wisconsin against proposed coll...,"health-care,labor,state-budget",republican-party-wisconsin,unknown,Wisconsin,republican,at protests in wisconsin against proposed coll...,141,...,republican-party-wisconsin,republican party wisconsin,unknown,2482,unknown,unknown,wisconsin,648,republican,3947


,id,statement,subject,speaker,speaker_job,state_info,party_affiliation,statement_clean,statement_length,statement_word_count,...,speaker_grouped,speaker_grouped_clean,speaker_job_clean,speaker_job_freq,speaker_job_grouped,speaker_job_grouped_clean,state_info_clean,state_info_freq,party_affiliation_clean,party_affiliation_freq
0,dc32e5ffa8b,Five members of [the Common Cause Georgia] boa...,"campaign-finance,ethics,government-regulation",kasim-reed,unknown,unknown,democrat,five members of the common cause georgia board...,89,12,...,kasim-reed,kasim reed,unknown,2482.0,unknown,unknown,unknown,1930.0,democrat,2898
1,aa49bb41cab,Theres no negative advertising in my campaign ...,elections,bill-mccollum,unknown,Florida,republican,theres no negative advertising in my campaign ...,53,9,...,bill-mccollum,bill mccollum,unknown,2482.0,unknown,unknown,florida,853.0,republican,3947
2,dddc8d12ac1,Leticia Van de Putte voted to give illegal imm...,"health-care,immigration,public-health",dan-patrick,Lieutenant governor-elect,Texas,republican,leticia van de putte voted to give illegal imm...,141,23,...,dan-patrick,dan patrick,lieutenant governor elect,18.0,Lieutenant governor-elect,lieutenant governor elect,texas,879.0,republican,3947
3,bcfe8f51667,Fiorinas plan would mean slashing Social Secur...,"federal-budget,medicare,social-security",barbara-boxer,U.S. Senator,California,democrat,fiorinas plan would mean slashing social secur...,63,9,...,barbara-boxer,barbara boxer,u s senator,391.0,U.S. Senator,u s senator,california,121.0,democrat,2898
4,eedbbaff5ab,"By the end of his first term, President Obama ...","federal-budget,new-hampshire-2012",mitt-romney,Former governor,Massachusetts,republican,by the end of his first term president obama w...,115,22,...,mitt-romney,mitt romney,former governor,142.0,Former governor,former governor,massachusetts,167.0,republican,3947
